In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import pickle

from sklearn.preprocessing import OrdinalEncoder

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [3]:
df = pd.read_csv(r'C:\Project\EV station website\Processed datasets\EV_usage_for_ML.csv')

In [ ]:
df.head()

In [ ]:
df = df.drop(columns=['user_income_level.1'])

In [ ]:
df.shape

In [ ]:
df['battery_health_pct'].plot(kind='box')
plt.show()

In [ ]:
df.dtypes

In [ ]:
df.select_dtypes(include=['str']).columns

In [ ]:
oe = OrdinalEncoder()

In [ ]:
cols = ['vehicle_type', 'charging_type', 'weather_condition', 'user_income_level']

df[cols] = oe.fit_transform(df[cols])

In [ ]:
y = df['battery_health_pct']
X = df.drop('battery_health_pct', axis=1)

In [ ]:
df = df[
    (df['battery_health_pct'] >= 0) &
    (df['battery_health_pct'] <= 100)
]

In [ ]:
df.info()
df.isnull().sum()
df.duplicated().sum()
df.describe()

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('battery_health_pct', axis=1)
y = df['battery_health_pct']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.dummy import DummyRegressor

dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)

dummy_pred = dummy.predict(X_test)

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print("MAE:", mean_absolute_error(y_test, pred))
    print("MSE:", mean_squared_error(y_test, pred))
    print("R2 Score:", r2_score(y_test, pred))

### Linear regression

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
lr = LinearRegression()

evaluate_model(
    lr,
    X_train_scaled, X_test_scaled,
    y_train, y_test
)

### Ridge

In [ ]:
ridge = Ridge(alpha=1.0)

evaluate_model(
    ridge,
    X_train_scaled, X_test_scaled,
    y_train, y_test
)

## Lasso

In [ ]:
lasso = Lasso(alpha=0.1)

evaluate_model(
    lasso,
    X_train_scaled, X_test_scaled,
    y_train, y_test
)

### Elastic net

In [ ]:
elastic = ElasticNet(
    alpha=0.1,
    l1_ratio=0.5,
    random_state=42
)

evaluate_model(
    elastic,
    X_train_scaled, X_test_scaled,
    y_train, y_test
)

### Decision Tree regressor

In [ ]:
dtr = DecisionTreeRegressor(
    random_state=42
)

evaluate_model(
    dtr,
    X_train, X_test,
    y_train, y_test
)

### Random Forest

In [ ]:
rfr = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

evaluate_model(
    rfr,
    X_train, X_test,
    y_train, y_test
)

In [ ]:
xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

evaluate_model(
    xgb,
    X_train, X_test,
    y_train, y_test
)

In [ ]:
results = []

models = {
    "Linear Regression": (lr, X_train_scaled, X_test_scaled),
    "Ridge": (ridge, X_train_scaled, X_test_scaled),
    "Lasso": (lasso, X_train_scaled, X_test_scaled),
    "Elastic Net": (elastic, X_train_scaled, X_test_scaled),
    "Decision Tree": (dtr, X_train, X_test),
    "Random Forest": (rfr, X_train, X_test),
    "XGBoost": (xgb, X_train, X_test)
}

for name, (model, Xtr, Xte) in models.items():
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "MSE": mean_squared_error(y_test, pred),
        "R2": r2_score(y_test, pred)
    })

results_df = pd.DataFrame(results)

print(results_df.sort_values("R2", ascending=False))

In [ ]:
rf_train_pred = rfr.predict(X_train)
rfr_pred = rfr.predict(X_test)

print("Random Forest Train R2:",
      r2_score(y_train, rf_train_pred))

print("Random Forest Test R2:",
      r2_score(y_test, rfr_pred))

In [ ]:
xgb_train_pred = xgb.predict(X_train)
xgb_pred = xgb.predict(X_test)

print("XGBoost Train R2:",
      r2_score(y_train, xgb_train_pred))

print("XGBoost Test R2:",
      r2_score(y_test, xgb_pred))

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rfr = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42
)

# Train
rfr.fit(X_train, y_train)

# Predictions
rf_train_pred = rfr.predict(X_train)
rf_test_pred = rfr.predict(X_test)

# Evaluation
print("TRAIN")
print("MAE:", mean_absolute_error(y_train, rf_train_pred))
print("MSE:", mean_squared_error(y_train, rf_train_pred))
print("R2:", r2_score(y_train, rf_train_pred))

print("\nTEST")
print("MAE:", mean_absolute_error(y_test, rf_test_pred))
print("MSE:", mean_squared_error(y_test, rf_test_pred))
print("R2:", r2_score(y_test, rf_test_pred))

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    rfr,
    X_train,
    y_train,
    cv=5,
    scoring='r2'
)

print("CV R2 scores:", cv_scores)
print("Mean CV R2:", cv_scores.mean())

### Tunned RandomForest

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [5, 8, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': ['sqrt', 'log2', 0.5, 1.0]
}

rf = RandomForestRegressor(
    random_state=42
)

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=30,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best parameters:")
print(random_search.best_params_)

print("\nBest CV R2:")
print(random_search.best_score_)

In [ ]:
best_rf = random_search.best_estimator_

In [ ]:
best_rf_pred = best_rf.predict(X_test)

print("Test MAE:",
      mean_absolute_error(y_test, best_rf_pred))

print("Test MSE:",
      mean_squared_error(y_test, best_rf_pred))

print("Test R2:",
      r2_score(y_test, best_rf_pred))

In [ ]:
best_rf_train_pred = best_rf.predict(X_train)

print("Train R2:",
      r2_score(y_train, best_rf_train_pred))

print("Test R2:",
      r2_score(y_test, best_rf_pred))

In [ ]:
print(random_search.best_params_)

In [ ]:
importance = pd.Series(
    best_rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance)

In [ ]:
cv_scores_tuned = cross_val_score(
    best_rf,
    X_train,
    y_train,
    cv=5,
    scoring='r2'
)

print("Tuned RF CV scores:", cv_scores_tuned)
print("Mean Tuned RF CV R2:", cv_scores_tuned.mean())

In [ ]:
df[['battery_health_pct', 'range_anxiety_risk']].corr()

In [ ]:
df[['battery_health_pct', 'range_anxiety_risk']].head(20)

In [ ]:
df.groupby('range_anxiety_risk')['battery_health_pct'].agg(
    ['count', 'mean', 'min', 'max']
)

In [ ]:
df.corr(numeric_only=True)['range_anxiety_risk'].sort_values(ascending=False)

### Tunned XGBoost

In [ ]:
param_grid_xgb = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'max_depth': [3, 4, 5, 6, 8],
    'min_child_weight': [1, 3, 5, 10],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

In [ ]:
xgb_model = XGBRegressor(
    objective='reg:squarederror',
    random_state=42
)

xgb_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid_xgb,
    n_iter=30,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

xgb_search.fit(X_train, y_train)

In [ ]:
print("Best Parameters:")
print(xgb_search.best_params_)

print("\nBest CV R2:")
print(xgb_search.best_score_)

In [ ]:
best_xgb = xgb_search.best_estimator_

In [ ]:
best_xgb_pred = best_xgb.predict(X_test)

print("Test MAE:",
      mean_absolute_error(y_test, best_xgb_pred))

print("Test MSE:",
      mean_squared_error(y_test, best_xgb_pred))

print("Test R2:",
      r2_score(y_test, best_xgb_pred))

In [ ]:
xgb_train_pred = best_xgb.predict(X_train)

print("Train R2:",
      r2_score(y_train, xgb_train_pred))

print("Test R2:",
      r2_score(y_test, best_xgb_pred))

In [ ]:
xgb_cv_scores = cross_val_score(
    best_xgb,
    X_train,
    y_train,
    cv=5,
    scoring='r2'
)

print("XGBoost CV scores:", xgb_cv_scores)
print("Mean XGBoost CV R2:", xgb_cv_scores.mean())

In [ ]:
importance_xgb = pd.Series(
    best_xgb.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance_xgb)

In [ ]:
print("RANDOM FOREST")
print("CV R2:", cv_scores_tuned.mean())
print("Test R2:", r2_score(y_test, best_rf_pred))
print("Test MAE:", mean_absolute_error(y_test, best_rf_pred))
print("Test MSE:", mean_squared_error(y_test, best_rf_pred))

print("\nXGBOOST")
print("CV R2:", xgb_cv_scores.mean())
print("Test R2:", r2_score(y_test, best_xgb_pred))
print("Test MAE:", mean_absolute_error(y_test, best_xgb_pred))
print("Test MSE:", mean_squared_error(y_test, best_xgb_pred))

In [ ]:
X_train.columns

In [ ]:
fh = open(r'C:\Project\EV station website\pickle files\battery_health.pkl','wb')
pickle.dump(xgb_search, fh)
fh.close()